In [14]:
# Peristence and Streaming
from langgraph.graph import StateGraph, END 
from typing import TypedDict, Annotated 
import operator 

from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_ollama import ChatOllama

# tool library
from langchain_core.tools import tool
from ddgs import DDGS
llm = "qwen3.8:27b"

In [15]:
# tools 
@tool
def get_weather(city: str) -> str:
    """Get current weather conditions for a single city."""
    import requests
    resp = requests.get(f"https://wttr.in/{city}?format=3")
    return resp.text
    
@tool
def get_search_query(query: str) -> str: 
    """Searches web for a query and returns search results."""
    try: 
        results = DDGS().text(query, max_results=15)
    except Exception as e:
        return f"Error calling tool: {e}"
    if not results:
        return "No results found" 
    return "\n\n".join(
        f"Title: {r['title']}\nURl: {r['href']}\n Body: {r['body']}"
        for r in results
    )


In [16]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [17]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)

In [18]:
class Agent: 
    def __init__(self, model, tools, checkpointer, system=""): 
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_ollama)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True:"action", False: END})
        graph.add_edge("action", "llm") 
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools} 
        self.model = model.bind_tools(tools) 
        
    def call_ollama(self, state:AgentState): 
        messages = state['messages'] 
        if self.system: 
            messages = [SystemMessage(content=self.system)] + messages 
        message = self.model.invoke(messages) 
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [19]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = ChatOllama(model=llm)  
abot = Agent(model, [get_weather, get_search_query], system=prompt, checkpointer=memory)

In [20]:
messages = [HumanMessage(content="What is the weather in sf")]

In [21]:
thread = {"configurable":{"thread_id":"1"}}

In [22]:
for event in abot.graph.stream({"messages":messages}, thread): 
    for v in event.values(): 
        print(v['messages'])

[AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-10-15T05:55:40.793530819Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4024596783, 'load_duration': 3474087078, 'prompt_eval_count': 402, 'prompt_eval_duration': 214885000, 'eval_count': 60, 'eval_duration': 333041000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a13e21-5ec0-7561-8a6f-e97a8166b708-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '0b59bd6f-3435-454b-b2cf-4d0fa0c24e7e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 402, 'output_tokens': 60, 'total_tokens': 462})]
Calling: {'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '0b59bd6f-3435-454b-b2cf-4d0fa0c24e7e', 'type': 'tool_call'}
Back to the model!
[ToolMessage(content='San Francisco: ☀️  +61°F\n', name='get_weather', tool_call_id='0b59bd6f-3435-454b-b2cf-4d0fa0c24e7e')]
[AIMessag

In [23]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable":{"thread_id":"1"}} # this ensures we are continuing same convo
for event in abot.graph.stream({"messages":messages}, thread): 
    for v in event.values(): 
        print(v['messages'])

[AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-10-15T05:55:54.411680462Z', 'done': True, 'done_reason': 'stop', 'total_duration': 342079678, 'load_duration': 1856670, 'prompt_eval_count': 500, 'prompt_eval_duration': 136457000, 'eval_count': 43, 'eval_duration': 199858000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a13e21-a254-71d0-a955-7ec891f94139-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Los Angeles'}, 'id': '39ac403a-44c6-4fe5-9bb0-ed79d30a0733', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 500, 'output_tokens': 43, 'total_tokens': 543})]
Calling: {'name': 'get_weather', 'args': {'city': 'Los Angeles'}, 'id': '39ac403a-44c6-4fe5-9bb0-ed79d30a0733', 'type': 'tool_call'}
Back to the model!
[ToolMessage(content='Los Angeles: ☀️  +73°F\n', name='get_weather', tool_call_id='39ac403a-44c6-4fe5-9bb0-ed79d30a0733')]
[AIMessage(content=

In [24]:
messages = [HumanMessage(content="Which is warmer?")]
thread = {"configurable":{"thread_id":"1"}} # this ensures we are continuing same convo
for event in abot.graph.stream({"messages":messages}, thread): 
    for v in event.values(): 
        print(v['messages'])

[AIMessage(content="**Los Angeles** is warmer at 73°F, compared to San Francisco's 61°F — a 12-degree difference!", additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-10-15T05:56:37.906233253Z', 'done': True, 'done_reason': 'stop', 'total_duration': 531315756, 'load_duration': 1830740, 'prompt_eval_count': 600, 'prompt_eval_duration': 149109000, 'eval_count': 71, 'eval_duration': 376807000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a13e22-4b7c-7c03-bc80-a5aa460845dd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 600, 'output_tokens': 71, 'total_tokens': 671})]


In [25]:
!uv pip install aiosqlite

Using Python 3.12.3 environment at: /home/acm/envs/vllmvenv
Audited 1 package in 1ms


In [26]:
import aiosqlite
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

conn = aiosqlite.connect(":memory:", check_same_thread=False)
memory = AsyncSqliteSaver(conn)
abot = Agent(model, [get_weather, get_search_query], system=prompt, checkpointer=memory)

In [28]:
messages = [HumanMessage(content="What is the weather in sf")]

thread = {"configurable":{"thread_id":"4"}}
async for event in abot.graph.astream_events({"messages": messages}, thread): 
    kind = event["event"] 
    if kind == "on_chat_model_stream": 
        content = event["data"]["chunk"].content
        if content:
            print(content, end="|")

Calling: {'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'c39b7507-6104-40c5-8c89-020270556e7d', 'type': 'tool_call'}
Back to the model!
The| weather| in| San| Francisco| is| currently| **|s|unny| ☀|️| at| 6|1|°F|**.| Enjoy| the| nice| weather|!|